# **Car Price Prediction**

In [7]:
# ── Imports ──────────────────────────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# **Load Dataset**

In [8]:
df = pd.read_csv("/content/Car data.csv")

# **View Dataset**

In [9]:
print(df.head())

print("\nShape:", df.shape)

print("\nColumns:")
print(df.columns)

               Name  Min Price (Lakh)  Max Price (Lakh)  Range (kmpl)    CC  \
0     Mahindra Thar             11.35             17.60         15.20  1497   
1      Maruti Swift              6.49              9.60         24.80  1197   
2  Mahindra Scorpio             13.62             17.42         22.00  2184   
3        Tata Punch              6.13             10.20         20.09  1199   
4     Hyundai Creta             11.00             20.15         17.40  1497   

   Seats  Variants     Type  Ex-Showroom Price       RTO  Insurance  Other  \
0      4        19   Petrol            1430000  143000.0    84367.0  14300   
1      5        11   Petrol             649000   46240.0    27791.0   5485   
2      7         4  Diesel             1361600  175000.0    85943.0  27532   
3      5        25   Petrol             612900   42903.0    35311.0      0   
4      5        28   Petrol            1099900  116863.0    45300.0  11599   

  Onroad Price  
0    16,71,667  
1     7,28,516  
2    

# **Check Missing Values**

In [10]:
print(df.isnull().sum())

Name                 0
Min Price (Lakh)     0
Max Price (Lakh)     0
Range (kmpl)         0
CC                   0
Seats                0
Variants             0
Type                 0
Ex-Showroom Price    0
RTO                  0
Insurance            0
Other                0
Onroad Price         0
dtype: int64


# **Clean Currency Columns**

In [11]:
price_columns = [
    "Ex-Showroom Price",
    "RTO",
    "Insurance",
    "Other",
    "Onroad Price"
]

for col in price_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .astype(float)
    )

# **Define Features and Target**

In [12]:
X = df.drop("Onroad Price", axis=1)

y = df["Onroad Price"]

# **Identify Numerical and Categorical Columns**

In [13]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("Numeric Columns:")
print(numeric_features)

print("\nCategorical Columns:")
print(categorical_features)

Numeric Columns:
Index(['Min Price (Lakh)', 'Max Price (Lakh)', 'Range (kmpl)', 'CC', 'Seats',
       'Variants', 'Ex-Showroom Price', 'RTO', 'Insurance', 'Other'],
      dtype='object')

Categorical Columns:
Index(['Name', 'Type'], dtype='object')


# **Numerical Pipeline**

In [14]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# **Categorical Pipeline**

In [15]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# **Combine Preprocessing**

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),

        ("cat", categorical_transformer, categorical_features)
    ]
)

# **Build ML Model Pipeline**

In [17]:
model = Pipeline(steps=[

    ("preprocessor", preprocessor),

    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# **Split Train and Test Data**

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (16, 12)
Testing Shape: (5, 12)


# **Train the Model**

In [19]:
model.fit(X_train, y_train)

print("Model Training Completed!")

Model Training Completed!


# **Make Predictions**

In [20]:
y_pred = model.predict(X_test)

# **Evaluate Model**

In [21]:
mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)

print("R2 Score:", r2)

MAE: 87644.16700000002
R2 Score: 0.9211971106843113


# **Compare Actual vs Predicted**

In [22]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

print(results)

   Actual Price  Predicted Price
0     1671667.0      1485737.630
1      698009.0       798856.130
2      751770.0       802917.465
3      728516.0       805970.145
4      841750.0       864592.725


# **Predict New Car Price**

In [23]:
sample = pd.DataFrame({
    'Name': ['Hyundai Creta'],
    'Min Price (Lakh)': [12],
    'Max Price (Lakh)': [18],
    'Range (kmpl)': [20],
    'CC': [1497],
    'Seats': [5],
    'Variants': [8],
    'Type': ['SUV'],
    'Ex-Showroom Price': [1500000],
    'RTO': [150000],
    'Insurance': [60000],
    'Other': [20000]
})

# **Predict Sample Price**

In [24]:
predicted_price = model.predict(sample)

print("Predicted Onroad Price: ₹", round(predicted_price[0], 2))

Predicted Onroad Price: ₹ 1524104.64
